In [2]:
#!/usr/bin/env python
"""
Ultra-simple script to test Wind_Evo at the first integration step.
"""

import numpy as np
import sys
import os

# sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from multiphasegalacticwind import WindModel
from multiphasegalacticwind.constants import *
from multiphasegalacticwind.core_physics import Wind_Evo, Hot_Wind_Evo
from multiphasegalacticwind.cooling import get_cooling_interpolator

# Create model with your exact parameters
model = WindModel(
    # Galaxy properties
    v_circ=0.00001,           # km/s, circular velocity (Milky Way-like)
    redshift=0.0,

    # Wind launch properties
    SFR = 1.0,               # Msun/yr, star formation rate
    eta_M = 1.0,              # hot phase mass loading
    eta_M_cold = 1e-12,         # cold phase mass loading
    eta_E = 0.01,              # energy loading

    # Sonic point
    r_star_kpc=0.3,         # kpc (= 300 pc)

    # Cloud properties
    cloud_mass_range=(10, 1e6),    # Msun
    cloud_alpha=2.0,               # power law slope dN/dM ∝ M^-α
    N_cloud_species=11,             # 11 cloud mass bins
    T_cl=1e4,                      # K, cloud temperature
    v_cloud_init=3.0,              # km/s, initial cloud velocity
    M_cloud_min=1e-1*Msun,          # Msun, minimum cloud mass
    cloud_radial_offset = 1e-6,

    # Integration settings
    r_max_kpc=30.0,        # kpc, maximum radius
    rtol=1e-12,              # relaxed tolerance for efficiency
    atol=1e-12,
    sonic_transition_tolerance = 1e-2,
    sonic_point_offset = 1e-6,
    Cooling_Factor = 0
)

print("=" * 80)
print("TESTING FIRST INTEGRATION STEP")
print("=" * 80)

# Get initial conditions (exactly as WindModel.run() does)
r_star = model.r_star_kpc * kpc
v_star_cgs = model.v_star * 1e5  # km/s to cm/s
rho_star = model.rho_star
P_star = model.P_star
v_circ_cgs = model.v_circ * 1e5

print(f"\nInitial conditions at r = {model.r_star_kpc} kpc:")
print(f"  v_star = {model.v_star:.3f} km/s")
print(f"  rho_star = {rho_star:.3e} g/cm^3")
print(f"  P_star = {P_star:.3e} dyne/cm^2")
print(f"  T_star = {model.T_star:.3e} K")

# Calculate Mach number
cs0 = np.sqrt(gamma * P_star / rho_star)
Mach0 = v_star_cgs / cs0
print(f"  cs0 = {cs0/1e5:.3f} km/s")
print(f"  Mach0 = {Mach0:.10f}")

# Build state vector (exactly as WindModel.run() does)
N = model.N_cloud_species
y0 = np.zeros(4 + 3*N)
y0[0] = v_star_cgs
y0[1] = rho_star
y0[2] = P_star
y0[3] = rho_star * model.config.Z_hot_over_Z_solar * Z_solar
y0[4:4+N] = model.M_cloud0  # Already in grams
y0[4+N:4+2*N] = model.config.v_cloud_init * 1e5  # cm/s
y0[4+2*N:] = model.config.Z_cloud_over_Z_solar * Z_solar

# Calculate source terms (exactly as WindModel.run() does)
SFR_cgs = model.SFR * Msun / yr
Edot = model.eta_E * (model.config.E_SN / (model.config.mstar * Msun)) * SFR_cgs
Mdot = model.eta_M * SFR_cgs
r0 = r_star
source_volume = 4./3. * np.pi * r0**3
Edot_per_Vol = Edot / source_volume
Mdot_per_Vol = Mdot / source_volume

print(f"\nSource terms:")
print(f"  Edot = {Edot:.3e} erg/s")
print(f"  Mdot = {Mdot:.3e} g/s")
print(f"  Edot_per_Vol = {Edot_per_Vol:.3e} erg/s/cm^3")
print(f"  Mdot_per_Vol = {Mdot_per_Vol:.3e} g/s/cm^3")

# Get cooling interpolator
cooling_interpolator = get_cooling_interpolator(
    model.config.mu, model.config.Z_hot_over_Z_solar, model.config.redshift
)

# Build parameters tuple (exactly as WindModel.run() does)
params = (
    v_circ_cgs,
    model.Ndot_cloud0,
    model.config.T_cl,
    model.config.cold_cloud_injection_radial_extent_frac,
    model.config.cold_cloud_injection_radial_power,
    model.config.to_dict(),
    r0,
    Edot_per_Vol,
    Mdot_per_Vol,
    cooling_interpolator
)

print("\n" + "-" * 80)
print("CALLING Wind_Evo AT INITIAL POSITION")
print("-" * 80)

# Call Wind_Evo at initial position
r_initial = r_star
derivatives = Wind_Evo(r_initial, y0, params)

print(f"\nDerivatives at r = {r_initial/kpc:.6f} kpc:")
print(f"  dv/dr = {derivatives[0]:.3e} (cm/s)/cm")
print(f"  drho/dr = {derivatives[1]:.3e} (g/cm^3)/cm")
print(f"  dP/dr = {derivatives[2]:.3e} (dyne/cm^2)/cm")
print(f"  drhoZ/dr = {derivatives[3]:.3e} (g/cm^3)/cm")

# What would happen after a tiny step?
dr = 0.001 * r_star  # 0.1% of initial radius
v_new = y0[0] + derivatives[0] * dr
rho_new = y0[1] + derivatives[1] * dr
P_new = y0[2] + derivatives[2] * dr

print(f"\nAfter tiny step dr = {dr/kpc:.6f} kpc:")
print(f"  v_new = {v_new/1e5:.3f} km/s (change: {(v_new-y0[0])/y0[0]*100:.2f}%)")
print(f"  rho_new = {rho_new:.3e} g/cm^3 (change: {(rho_new-y0[1])/y0[1]*100:.2f}%)")
print(f"  P_new = {P_new:.3e} dyne/cm^2 (change: {(P_new-y0[2])/y0[2]*100:.2f}%)")

if P_new < 0:
    print("  ⚠️ PRESSURE GOES NEGATIVE!")
if rho_new < 0:
    print("  ⚠️ DENSITY GOES NEGATIVE!")

# Calculate new Mach number
cs_new = np.sqrt(gamma * P_new / rho_new) if P_new > 0 and rho_new > 0 else 0
Mach_new = v_new / cs_new if cs_new > 0 else np.inf
print(f"  Mach_new = {Mach_new:.4f}")

print("\n" + "-" * 80)
print("COMPARING WITH Hot_Wind_Evo (HOT-ONLY)")
print("-" * 80)

# Build state vector for hot-only (just first 3 components)
y0_hot = y0[:3].copy()

# Parameters for hot wind with source terms
params_hot = (v_circ_cgs, True, r0, Edot_per_Vol, Mdot_per_Vol)

# Call Hot_Wind_Evo at initial position
derivatives_hot = Hot_Wind_Evo(r_initial, y0_hot, params_hot)

print(f"\nHot-only derivatives at r = {r_initial/kpc:.6f} kpc:")
print(f"  dv/dr = {derivatives_hot[0]:.3e} (cm/s)/cm")
print(f"  drho/dr = {derivatives_hot[1]:.3e} (g/cm^3)/cm")
print(f"  dP/dr = {derivatives_hot[2]:.3e} (dyne/cm^2)/cm")

# What would happen after a tiny step?
v_new_hot = y0_hot[0] + derivatives_hot[0] * dr
rho_new_hot = y0_hot[1] + derivatives_hot[1] * dr
P_new_hot = y0_hot[2] + derivatives_hot[2] * dr

print(f"\nHot-only after tiny step dr = {dr/kpc:.6f} kpc:")
print(f"  v_new = {v_new_hot/1e5:.3f} km/s (change: {(v_new_hot-y0_hot[0])/y0_hot[0]*100:.2f}%)")
print(f"  rho_new = {rho_new_hot:.3e} g/cm^3 (change: {(rho_new_hot-y0_hot[1])/y0_hot[1]*100:.2f}%)")
print(f"  P_new = {P_new_hot:.3e} dyne/cm^2 (change: {(P_new_hot-y0_hot[2])/y0_hot[2]*100:.2f}%)")

if P_new_hot < 0:
    print("  ⚠️ PRESSURE GOES NEGATIVE!")
if rho_new_hot < 0:
    print("  ⚠️ DENSITY GOES NEGATIVE!")

print("\n" + "-" * 80)
print("COMPARISON: Wind_Evo vs Hot_Wind_Evo")
print("-" * 80)

print("\nRatio of derivatives (Wind_Evo / Hot_Wind_Evo):")
print(f"  dv/dr ratio = {derivatives[0]/derivatives_hot[0]:.3f}")
print(f"  drho/dr ratio = {derivatives[1]/derivatives_hot[1]:.3f}")
print(f"  dP/dr ratio = {derivatives[2]/derivatives_hot[2]:.3f}")

print("\nDifference in final values after step:")
print(f"  Δv = {(v_new - v_new_hot)/1e5:.3f} km/s")
print(f"  Δrho = {rho_new - rho_new_hot:.3e} g/cm^3")
print(f"  ΔP = {P_new - P_new_hot:.3e} dyne/cm^2")

print("\n" + "=" * 80)
print("DIAGNOSTICS")
print("=" * 80)

# Check if we're in the singular denominator region
sonic_denom = 1.0 - (1.0/Mach0**2)
print(f"\nSonic denominator = 1 - 1/Mach^2 = {sonic_denom:.10f}")
if abs(sonic_denom) < 1e-6:
    print("  ⚠️ Very close to singularity!")

# Check if we're at the source boundary
print(f"\nSource region radius r0 = {r0/kpc:.6f} kpc")
print(f"Initial radius r_star = {r_star/kpc:.6f} kpc")
if abs(r_star - r0) < 1e-10:
    print("  ⚠️ Starting exactly at source boundary!")
    print("  Note: Source terms only apply when r < r0")
else:
    print(f"  Offset = {(r_star - r0)/pc:.3f} pc")

# Check cloud properties
print(f"\nCloud initial velocities: {model.config.v_cloud_init} km/s")
print(f"Cloud masses (Msun): {model.M_cloud0/Msun}")
print(f"cloud_radial_offset = {model.config.cloud_radial_offset}")

print("\nTo modify parameters, edit this script and re-run!")

TESTING FIRST INTEGRATION STEP

Initial conditions at r = 0.3 kpc:
  v_star = 50.000 km/s
  rho_star = 1.178e-24 g/cm^3
  P_star = 1.766e-11 dyne/cm^2
  T_star = 1.127e+05 K
  cs0 = 50.000 km/s
  Mach0 = 1.0000010000

Source terms:
  Edot = 3.171e+39 erg/s
  Mdot = 6.342e+25 g/s
  Edot_per_Vol = 9.540e-25 erg/s/cm^3
  Mdot_per_Vol = 1.908e-38 g/s/cm^3

--------------------------------------------------------------------------------
CALLING Wind_Evo AT INITIAL POSITION
--------------------------------------------------------------------------------

Derivatives at r = 0.300000 kpc:
  dv/dr = 5.401e-09 (cm/s)/cm
  drho/dr = -1.272e-39 (g/cm^3)/cm
  dP/dr = -3.180e-26 (dyne/cm^2)/cm
  drhoZ/dr = -8.045e-42 (g/cm^3)/cm

After tiny step dr = 0.000300 kpc:
  v_new = 50049.994 km/s (change: 99999.95%)
  rho_new = -1.176e-21 g/cm^3 (change: -100000.15%)
  P_new = -2.942e-08 dyne/cm^2 (change: -166666.92%)
  ⚠️ PRESSURE GOES NEGATIVE!
  ⚠️ DENSITY GOES NEGATIVE!
  Mach_new = inf

--------------

In [ ]:
derivatives